# Week 2 – Agribusiness Data Modeling and SQL Query Development

**Author:** Sayantani Das  
**Department:** B.Tech CSE  
**Database:** MySQL 8.0+  
**Environment:** Google Colab

This notebook installs MySQL in the temporary Colab runtime, imports the project SQL script, verifies the schema, and executes analytical queries with results displayed as pandas tables.

> **Important:** Colab runtimes are temporary. If the runtime disconnects or resets, rerun the setup cells and upload the SQL script again.

## 1. Install MySQL Server
Run this cell once in a fresh Colab runtime. Installation may take a few minutes.

In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y mysql-server

## 2. Start MySQL Server
Start the server and check the installed version.

In [ ]:
!service mysql start
!mysql --version

## 3. Upload and execute the project SQL script

Download `agribusiness_database.sql` from the project deliverables and upload it when prompted. The script creates the database, all 10 tables, sample records, and contains the 15 analytical queries.

If you have already uploaded the file to the current Colab session, you can skip the upload and set `sql_file` to its path.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select agribusiness_database.sql

sql_file = next((name for name in uploaded if name.lower().endswith(".sql")), None)
if sql_file is None:
    raise FileNotFoundError("Please upload agribusiness_database.sql")

print("Selected SQL file:", sql_file)

In [ ]:
# Execute the complete SQL script.
# The script creates agribusiness_db, tables, sample data and query definitions.
import subprocess

with open(sql_file, "rb") as sql_input:
    result = subprocess.run(
        ["sudo", "mysql"],
        stdin=sql_input,
        capture_output=True,
        text=True
    )

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("SQL script execution failed. Review the MySQL error above.")
else:
    print("SQL script executed successfully.")
    if result.stderr.strip():
        print(result.stderr)

## 4. Verify database and tables
The expected database is `agribusiness_db` and it should contain 10 tables.

In [ ]:
!sudo mysql -D agribusiness_db -e "SHOW TABLES;" 

## 5. Connect Python to MySQL
Install PyMySQL and create a connection so query outputs can be displayed as pandas DataFrames.

In [ ]:
!pip -q install pymysql

In [ ]:
import pymysql
import pandas as pd
from IPython.display import display

connection = pymysql.connect(
    unix_socket="/var/run/mysqld/mysqld.sock",
    user="root",
    database="agribusiness_db",
    cursorclass=pymysql.cursors.DictCursor
)

print("Successfully connected to Agribusiness Database!")

## 6. Run the 15 analytical SQL queries
Each query answers a practical agribusiness question. Run the following cell to display all result sets. You can also execute each query separately by copying it into a new cell.

In [ ]:
queries = {'Q1. Farm ownership and location': "SELECT f.farm_id, f.farm_name,\n       CONCAT(fr.first_name, ' ', fr.last_name) AS farmer_name,\n       f.location, f.total_area_ha\nFROM farms AS f\nJOIN farmers AS fr ON fr.farmer_id = f.farmer_id\nORDER BY f.farm_name;", 'Q2. Total harvested quantity by crop': 'SELECT c.crop_name, SUM(h.quantity_tonnes) AS total_harvest_tonnes\nFROM harvests h\nJOIN plantings p ON p.planting_id = h.planting_id\nJOIN crops c ON c.crop_id = p.crop_id\nGROUP BY c.crop_id, c.crop_name\nORDER BY total_harvest_tonnes DESC;', 'Q3. Yield per hectare by planting': 'SELECT p.planting_id, f.farm_name, fld.field_name, c.crop_name,\n       SUM(h.quantity_tonnes) AS harvested_tonnes, fld.area_ha,\n       ROUND(SUM(h.quantity_tonnes) / fld.area_ha, 2) AS yield_tonnes_per_ha\nFROM plantings p\nJOIN fields fld ON fld.field_id = p.field_id\nJOIN farms f ON f.farm_id = fld.farm_id\nJOIN crops c ON c.crop_id = p.crop_id\nJOIN harvests h ON h.planting_id = p.planting_id\nGROUP BY p.planting_id, f.farm_name, fld.field_name, c.crop_name, fld.area_ha\nORDER BY yield_tonnes_per_ha DESC;', 'Q4. Production by crop and season': 'SELECT s.season_name, c.crop_name,\n       SUM(h.quantity_tonnes) AS production_tonnes\nFROM harvests h\nJOIN plantings p ON p.planting_id = h.planting_id\nJOIN seasons s ON s.season_id = p.season_id\nJOIN crops c ON c.crop_id = p.crop_id\nGROUP BY s.season_id, s.season_name, c.crop_id, c.crop_name\nORDER BY s.start_date, production_tonnes DESC;', 'Q5. Revenue by farm': 'SELECT f.farm_name,\n       SUM(sa.quantity_tonnes * sa.unit_price) AS revenue\nFROM sales sa\nJOIN harvests h ON h.harvest_id = sa.harvest_id\nJOIN plantings p ON p.planting_id = h.planting_id\nJOIN fields fld ON fld.field_id = p.field_id\nJOIN farms f ON f.farm_id = fld.farm_id\nGROUP BY f.farm_id, f.farm_name\nORDER BY revenue DESC;', 'Q6. Expenses by farm and category': 'SELECT f.farm_name, e.expense_category,\n       SUM(e.amount) AS total_expense\nFROM expenses e\nJOIN farms f ON f.farm_id = e.farm_id\nGROUP BY f.farm_id, f.farm_name, e.expense_category\nORDER BY f.farm_name, total_expense DESC;', 'Q7. Simplified operating surplus by farm': 'WITH farm_revenue AS (\n    SELECT fld.farm_id, SUM(sa.quantity_tonnes * sa.unit_price) AS revenue\n    FROM sales sa\n    JOIN harvests h ON h.harvest_id = sa.harvest_id\n    JOIN plantings p ON p.planting_id = h.planting_id\n    JOIN fields fld ON fld.field_id = p.field_id\n    GROUP BY fld.farm_id\n),\nfarm_cost AS (\n    SELECT farm_id, SUM(amount) AS expenses\n    FROM expenses GROUP BY farm_id\n)\nSELECT f.farm_name, COALESCE(r.revenue,0) AS revenue,\n       COALESCE(c.expenses,0) AS expenses,\n       COALESCE(r.revenue,0)-COALESCE(c.expenses,0) AS operating_surplus\nFROM farms f\nLEFT JOIN farm_revenue r ON r.farm_id=f.farm_id\nLEFT JOIN farm_cost c ON c.farm_id=f.farm_id\nORDER BY operating_surplus DESC;', 'Q8. Monthly sales revenue': "SELECT DATE_FORMAT(sale_date,'%Y-%m') AS sales_month,\n       SUM(quantity_tonnes * unit_price) AS monthly_revenue\nFROM sales\nGROUP BY DATE_FORMAT(sale_date,'%Y-%m')\nORDER BY sales_month;", 'Q9. Best customers by purchase value': 'SELECT cu.customer_name,\n       SUM(sa.quantity_tonnes * sa.unit_price) AS purchase_value,\n       SUM(sa.quantity_tonnes) AS purchased_tonnes\nFROM customers cu\nJOIN sales sa ON sa.customer_id=cu.customer_id\nGROUP BY cu.customer_id, cu.customer_name\nORDER BY purchase_value DESC;', 'Q10. Unsold harvest quantity': 'SELECT h.harvest_id, c.crop_name, h.quantity_tonnes,\n       COALESCE(SUM(sa.quantity_tonnes),0) AS sold_tonnes,\n       h.quantity_tonnes-COALESCE(SUM(sa.quantity_tonnes),0) AS unsold_tonnes\nFROM harvests h\nJOIN plantings p ON p.planting_id=h.planting_id\nJOIN crops c ON c.crop_id=p.crop_id\nLEFT JOIN sales sa ON sa.harvest_id=h.harvest_id\nGROUP BY h.harvest_id,c.crop_name,h.quantity_tonnes\nHAVING unsold_tonnes>0\nORDER BY unsold_tonnes DESC;', 'Q11. Farm production ranking': 'WITH farm_production AS (\n    SELECT f.farm_id, f.farm_name, SUM(h.quantity_tonnes) AS total_tonnes\n    FROM farms f\n    JOIN fields fld ON fld.farm_id=f.farm_id\n    JOIN plantings p ON p.field_id=fld.field_id\n    JOIN harvests h ON h.planting_id=p.planting_id\n    GROUP BY f.farm_id,f.farm_name\n)\nSELECT farm_name,total_tonnes,\n       DENSE_RANK() OVER (ORDER BY total_tonnes DESC) AS production_rank\nFROM farm_production;', 'Q12. Average selling price per crop': 'SELECT c.crop_name,\n       ROUND(SUM(sa.quantity_tonnes*sa.unit_price) /\n             NULLIF(SUM(sa.quantity_tonnes),0),2) AS avg_price_per_tonne\nFROM sales sa\nJOIN harvests h ON h.harvest_id=sa.harvest_id\nJOIN plantings p ON p.planting_id=h.planting_id\nJOIN crops c ON c.crop_id=p.crop_id\nGROUP BY c.crop_id,c.crop_name\nORDER BY avg_price_per_tonne DESC;', 'Q13. Planned and growing plantings': "SELECT p.planting_id,f.farm_name,fld.field_name,c.crop_name,\n       p.sowing_date,p.expected_harvest_date,p.status\nFROM plantings p\nJOIN fields fld ON fld.field_id=p.field_id\nJOIN farms f ON f.farm_id=fld.farm_id\nJOIN crops c ON c.crop_id=p.crop_id\nWHERE p.status IN ('PLANNED','GROWING')\nORDER BY p.expected_harvest_date;", 'Q14. Monthly expense trend': "SELECT DATE_FORMAT(expense_date,'%Y-%m') AS expense_month,\n       SUM(amount) AS monthly_expense\nFROM expenses\nGROUP BY DATE_FORMAT(expense_date,'%Y-%m')\nORDER BY expense_month;", 'Q15. Top-producing crop in each season': 'WITH crop_season_yield AS (\n    SELECT s.season_name,c.crop_name,SUM(h.quantity_tonnes) AS total_tonnes\n    FROM harvests h\n    JOIN plantings p ON p.planting_id=h.planting_id\n    JOIN seasons s ON s.season_id=p.season_id\n    JOIN crops c ON c.crop_id=p.crop_id\n    GROUP BY s.season_id,s.season_name,c.crop_id,c.crop_name\n),\nranked AS (\n    SELECT *,DENSE_RANK() OVER\n      (PARTITION BY season_name ORDER BY total_tonnes DESC) AS crop_rank\n    FROM crop_season_yield\n)\nSELECT season_name,crop_name,total_tonnes\nFROM ranked WHERE crop_rank=1\nORDER BY season_name;'}

for title, query in queries.items():
    print("\\n" + "=" * 90)
    print(title)
    print("=" * 90)
    df = pd.read_sql_query(query, connection)
    display(df)


## 7. Close the database connection

Run this after completing the analysis.

In [ ]:
connection.close()
print("MySQL connection closed.")

## 8. Conclusion

The Agribusiness Farm Management Database organizes farmer, farm, field, crop, season, planting, harvest, expense, customer and sales information in a relational structure. The analytical queries demonstrate joins, grouping and aggregation, common table expressions, date-based reporting and window functions. The database supports crop-yield analysis, farm comparisons, revenue and expense monitoring, customer analysis and seasonal production decisions.

**Submission note:** Save this notebook as `Agribusiness_Week2_SQL_Project.ipynb` and submit it alongside the Word report, SQL script and README file.